In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2FLean_angle_detection_of_pole%2FUtility_pole_lean_angle_detection.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Calculate lean angle of a pole using code execution

This notebook uses the Gemini model to analyze lean angle of a pole. It fetches image URIs from a BigQuery table and then uses prompts to instruct the Gemini model to perform the following analyses:
- Detect the pole
- Run canny Edge detection model
- Run Hough line detection algorithm
- Output the angle as an integer



## Install required libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery
!pip install --upgrade google-genai

## Configuration

In [ ]:
PROJECT_ID = ''  # @param {type:"string"}
REGION = ''      # @param {type:"string"}
MODEL_ID = "gemini-2.5-flash" # @param {type:"string"}
BIGQUERY_DATASET_ID = '' # @param {type:"string"}
BIGQUERY_TABLE = "latest_observations" # @param {type:"string"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
LIMIT = 5 # @param {type:"integer"}

## Imports and Vertex AI Initialization

In [ ]:
import vertexai
from google.cloud import bigquery
from google import genai
from google.genai.types import (
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    HttpOptions,
    Part,
    Tool,
    ToolCodeExecution,
    SafetySetting,
)
import pandas as pd
import re
import json



## Initialize the vertex AI SDK

In [ ]:
vertexai.init(project=PROJECT_ID, location=REGION)
print("pandas, re, and json libraries have been successfully imported and Vertex AI initialized.")

## Fetch Image URIs from BigQuery

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  *
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE}`
   WHERE asset_type = "{ASSET_TYPE}"
LIMIT {LIMIT};
"""

try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    gcs_uris = [item.get("gcs_uri") for item in query_response_data if item.get("gcs_uri")]
    print(f"Successfully fetched {len(gcs_uris)} GCS URIs:")
    for uri in gcs_uris:
        print(uri)
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Define Analysis Prompts

In [ ]:
PROMPT = """You will be provided with an image for analysis.

Your task is to perform Canny edge detection and Hough angle calculations to determine the angle of a pole in the image. Follow these steps exactly:

1.  Load the Image: Access the provided image file.
2.  Convert to Grayscale: Convert the color image into a single-channel grayscale image.
3.  Apply Gaussian Blur: Apply a Gaussian blur to smooth the image and reduce noise.
4.  Filter for Vertical/Near-Vertical Lines: Identify lines that are likely to represent a pole. These are typically near-vertical lines.
5.  Perform Canny Edge Detection: Apply the Canny edge detection algorithm to find edges in the image. Use appropriate thresholds.
6.  Perform Hough Line Transform: Apply the Probabilistic Hough Line Transform to detect straight lines in the Canny edge-detected image.
7.  Calculate Pole Angle: From the detected pole lines, calculate the average angle of the pole relative to the vertical axis.
8.  Format the Output: Your final output must be ONLY the calculated angle, rounded to the nearest whole number (integer). Do not include any other text, explanation, or code."""

## Create vertex client

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Run Analysis

In [ ]:
# Initialize the genai Client and the code execution Tool

code_execution_tool = Tool(code_execution=ToolCodeExecution())

def analyze_image(gcs_uri: str, prompt: str) -> str:
    """Analyzes an image from GCS using the Gemini model with code execution, returning None on error."""
    try:
        contents = [
            prompt,
            Part(file_data={'file_uri': gcs_uri, 'mime_type': 'image/jpeg'})
        ]
        # Update the generate_content call to include the code execution tool
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=GenerateContentConfig(
            tools=[code_execution_tool],
            temperature=0,
              )
        )
        print("# Code:")
        print(response.executable_code)
        print("# Outcome:")
        print(response.code_execution_result)
        return response.text
    except Exception as e:
        print(f"Error analyzing '{gcs_uri}': {e}")
        return None

results = []

if 'query_response_data' in locals() and query_response_data:
    for item in query_response_data:
        asset_id = item.get("asset_id")
        uri = item.get("gcs_uri")

        if not uri or not asset_id:
            continue

        print(f"--- Analyzing {uri} ---")

        # Analyze image with the new prompt for pole angle
        pole_angle_response = analyze_image(uri, PROMPT)

        # Parse pole angle result
        pole_angle = None
        try:
            if pole_angle_response:
                pole_angle = int(float(re.search(r'[-+]?\d*\.?\d+', pole_angle_response).group()))
        except (ValueError, AttributeError, TypeError):
            pole_angle = None

        # Store results
        result_item = {
            "asset_id": asset_id,
            "pole_angle": pole_angle
        }
        results.append(result_item)

    # Create and display DataFrame
    if results:
        df = pd.DataFrame(results)
        df.set_index('asset_id', inplace=True)
        print("\n--- Analysis Results ---")
        display(df)
    else:
        print("No data to display.")

else:
    print("No GCS URIs were found to analyze.")